# [5.3] Mamba from Scratch - Exercises

**Core question:** how can a Mamba-style selective state-space layer remember one input, ignore distractors, expose the memory only when asked, and still give the same states under sequential, parallel, and chunked execution?

**Claim.** By the end of this notebook, you will have implemented every recurrence operation needed to show that an exact selective-memory organism has zero read error, while fixed dynamics and broken chunk state visibly fail.

<img src="../../instructions/assets/mamba_scan_contract.svg" width="820">


## Learning objectives

You will:

- discretize stable continuous-time SSM parameters into affine updates;
- implement the sequential recurrence as a hand-checkable reference;
- derive and implement an associative parallel prefix scan;
- carry recurrent state correctly across chunks;
- use input-dependent update and read gates on an exact ground-truth task;
- intervene on the recurrent state and interpret the downstream effect;
- compare the selective model with fixed-decay and chunk-reset controls.


## Setup

Everything in the main lesson runs on CPU. The tiny event sequence is an **exact model organism**: its gates are specified rather than trained, so its state has an unambiguous ground-truth meaning. Real Mamba layers learn finite, approximate gates from token representations; we return to that limitation at the end.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter5_modern_architectures"
section = "part3_mamba_from_scratch"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
exercises_dir = root_dir / chapter / "exercises"
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_mamba_from_scratch.tests as tests

EVENT_HOLD = 0
EVENT_WRITE = 1
EVENT_READ = 2
EVENT_ERASE = 3


@dataclass(frozen=True)
class SelectiveCopyCase:
    event_ids: t.Tensor
    values: t.Tensor
    labels: tuple[str, ...]
    read_positions: t.Tensor
    read_targets: t.Tensor


@dataclass(frozen=True)
class SelectiveCopyResult:
    case: SelectiveCopyCase
    a: t.Tensor
    b: t.Tensor
    c: t.Tensor
    sequential_states: t.Tensor
    parallel_states: t.Tensor
    chunked_states: t.Tensor
    selective_reads: t.Tensor
    fixed_decay_reads: t.Tensor
    reset_chunk_reads: t.Tensor
    ablated_reads: t.Tensor
    parity_max_abs_diff: float
    chunked_max_abs_diff: float
    selective_read_mae: float
    fixed_decay_read_mae: float
    reset_chunk_read_mae: float
    ablation_effect: float


def make_selective_copy_case(
    first_value: float = 1.0,
    second_value: float = -0.7,
    first_delay: int = 2,
    second_delay: int = 2,
    final_delay: int = 1,
    dtype: t.dtype = t.float64,
) -> SelectiveCopyCase:
    """Create an exact write-hold-read-erase sequence with known answers."""
    if min(first_delay, second_delay, final_delay) < 0:
        raise ValueError("delay lengths must be non-negative.")
    events = (
        [EVENT_WRITE]
        + [EVENT_HOLD] * first_delay
        + [EVENT_READ, EVENT_HOLD, EVENT_WRITE]
        + [EVENT_HOLD] * second_delay
        + [EVENT_READ, EVENT_ERASE]
        + [EVENT_HOLD] * final_delay
        + [EVENT_READ]
    )
    event_ids = t.tensor([events], dtype=t.long)
    values = t.linspace(-0.9, 0.9, len(events), dtype=dtype).unsqueeze(0)
    write_positions = event_ids[0].eq(EVENT_WRITE).nonzero(as_tuple=False).flatten()
    values[0, write_positions[0]] = first_value
    values[0, write_positions[1]] = second_value
    read_positions = event_ids[0].eq(EVENT_READ).nonzero(as_tuple=False).flatten()
    read_targets = t.tensor([first_value, second_value, 0.0], dtype=dtype)
    labels = []
    for position, event in enumerate(events):
        if event == EVENT_WRITE:
            labels.append(f"WRITE {values[0, position].item():+.1f}")
        elif event == EVENT_READ:
            labels.append("READ")
        elif event == EVENT_ERASE:
            labels.append("ERASE")
        else:
            labels.append("hold")
    return SelectiveCopyCase(event_ids, values, tuple(labels), read_positions, read_targets)


case = make_selective_copy_case()
print(f"{'pos':>3}  {'event':>10}  {'input value':>11}")
for position, (label, value) in enumerate(zip(case.labels, case.values[0])):
    print(f"{position:>3}  {label:>10}  {value.item():>+11.2f}")


## Cold open: one scalar of memory

The sequence contains ordinary distractor values, but only four event types matter:

- `WRITE`: replace the state with the attached value;
- `hold`: keep the previous state and ignore the distractor;
- `READ`: expose the state without changing it;
- `ERASE`: set the state to zero.

The three exact answers are `+1.0`, `-0.7`, and `0.0`. The concrete question is whether a selective recurrence can produce those answers while two plausible nonselective or incorrectly cached versions fail.


## Background: Mamba as input-conditioned affine updates

A diagonal state-space recurrence can be written elementwise as

$$h_t = a_t \odot h_{t-1} + b_t, \qquad y_t = C_t h_t.$$

In a Mamba layer, the token stream produces input-dependent $\Delta_t$, $B_t$, and $C_t$. With stable $A=-\exp(A_{\log})$, the common discretization used in this lesson is

$$a_t = \exp(\Delta_t A), \qquad b_t = \Delta_t B_t u_t.$$

This is the bridge from continuous-time SSM parameters to the affine scan. Once `a` and `b` exist, execution order is a separate correctness problem.


### Exercise 1 - discretize stable SSM parameters

Implement `_expand_bc` and `discretize_selective_scan`. The returned tensors should have shape `(batch, sequence, d_inner, d_state)`, and every value of `a` must lie in `(0, 1]`.

<details><summary>Expected output</summary>

```text
All tests in `test_discretize_selective_scan_shapes_and_stability` passed!
```
</details>

<details><summary>Help</summary>

Expand rank-3 `B` with a new `d_inner` axis. Form `A = -exp(A_log)`, then broadcast `delta` and `u` over `d_state`.
</details>

<details><summary>Solution</summary>

```python
A = -t.exp(A_log).to(dtype=u.dtype, device=u.device)
B_expanded = _expand_bc(B.to(dtype=u.dtype, device=u.device), u.shape[-1])
a = t.exp(delta.unsqueeze(-1) * A[None, None, :, :])
b = delta.unsqueeze(-1) * u.unsqueeze(-1) * B_expanded
return a, b
```
</details>


In [ ]:
def _expand_bc(param: t.Tensor, d_inner: int) -> t.Tensor:
    """Expand B from (batch, seq, d_state) across d_inner when needed."""
    raise NotImplementedError()


def discretize_selective_scan(
    u: t.Tensor,
    delta: t.Tensor,
    A_log: t.Tensor,
    B: t.Tensor,
) -> tuple[t.Tensor, t.Tensor]:
    """Return a_t and b_t for h_t = a_t * h_(t-1) + b_t."""
    raise NotImplementedError()


tests.test_discretize_selective_scan_shapes_and_stability(discretize_selective_scan)


### Exercise 2 - write the sequential reference

Implement the recurrence literally. Save every post-update state, because those states will become the first panel of the signature result.

<details><summary>Expected output</summary>

```text
All tests in `test_sequential_affine_scan_manual` passed!
```

The manual fixture starts from `2.0` and must return `[2.0, -0.5, 2.5]`.
</details>

<details><summary>Help</summary>

Sequence is axis 1. Initialize `state` with shape `b[:, 0].shape`, update it once per position, append each new state, then stack on axis 1.
</details>

<details><summary>Solution</summary>

```python
state = t.zeros_like(b[:, 0]) if initial_state is None else initial_state.to(b)
states = []
for position in range(a.shape[1]):
    state = a[:, position] * state + b[:, position]
    states.append(state)
return t.stack(states, dim=1)
```
</details>


In [ ]:
def sequential_affine_scan(
    a: t.Tensor,
    b: t.Tensor,
    initial_state: t.Tensor | None = None,
) -> t.Tensor:
    """Apply h_t = a_t * h_(t-1) + b_t from left to right."""
    raise NotImplementedError()


tests.test_sequential_affine_scan_manual(sequential_affine_scan)


### Exercise 3 - compose updates and scan in parallel

Each token represents an affine map `(a, b)`. Applying the right map after the left map gives

$$(a_R,b_R) \circ (a_L,b_L) = (a_Ra_L,\ a_Rb_L+b_R).$$

Implement this operation, then use offsets `1, 2, 4, ...` for an inclusive Hillis-Steele prefix scan. The test compares all states on random float64 recurrences, including a nonzero initial state.

<details><summary>Expected output</summary>

```text
All tests in `test_compose_affine_updates` passed!
All tests in `test_parallel_affine_scan_parity` passed!
```

Maximum sequential-versus-parallel error must be at most `1e-12`.
</details>

<details><summary>Help</summary>

Clone both prefix tensors at each offset. Compose `old[:, :-offset]` on the left with `old[:, offset:]` on the right. In-place reads from partially updated prefixes break the scan.
</details>

<details><summary>Solution</summary>

```python
def compose_affine_updates(a_left, b_left, a_right, b_right):
    return a_right * a_left, a_right * b_left + b_right

a_prefix, b_prefix = a.clone(), b.clone()
offset = 1
while offset < a.shape[1]:
    old_a, old_b = a_prefix.clone(), b_prefix.clone()
    a_prefix[:, offset:], b_prefix[:, offset:] = compose_affine_updates(
        old_a[:, :-offset], old_b[:, :-offset],
        old_a[:, offset:], old_b[:, offset:],
    )
    offset *= 2
```
</details>


In [ ]:
def compose_affine_updates(
    a_left: t.Tensor,
    b_left: t.Tensor,
    a_right: t.Tensor,
    b_right: t.Tensor,
) -> tuple[t.Tensor, t.Tensor]:
    """Compose the right update after the left update."""
    raise NotImplementedError()


def parallel_affine_scan(
    a: t.Tensor,
    b: t.Tensor,
    initial_state: t.Tensor | None = None,
) -> t.Tensor:
    """Inclusive Hillis-Steele scan over affine recurrence updates."""
    raise NotImplementedError()


tests.test_compose_affine_updates(compose_affine_updates)
tests.test_parallel_affine_scan_parity(sequential_affine_scan, parallel_affine_scan)


### Exercise 4 - carry state across chunks

Chunking should change memory use, not model behavior. Implement chunked sequential execution and pass the last state from one chunk into the next.

<details><summary>Expected output</summary>

```text
All tests in `test_chunked_affine_scan_state_carry` passed!
```

Chunk sizes `1`, `4`, `7`, and a chunk larger than the sequence all match bit-for-bit.
</details>

<details><summary>Help</summary>

After scanning one slice, set `state = chunk_states[:, -1]`. The `reset_each_chunk` branch is intentionally retained for a later negative control.
</details>

<details><summary>Solution</summary>

```python
for start in range(0, a.shape[1], chunk_size):
    stop = min(start + chunk_size, a.shape[1])
    if reset_each_chunk and start > 0:
        state = t.zeros_like(state)
    chunk_states = sequential_affine_scan(a[:, start:stop], b[:, start:stop], state)
    chunks.append(chunk_states)
    state = chunk_states[:, -1]
return t.cat(chunks, dim=1)
```
</details>


In [ ]:
def chunked_affine_scan(
    a: t.Tensor,
    b: t.Tensor,
    chunk_size: int,
    initial_state: t.Tensor | None = None,
    *,
    reset_each_chunk: bool = False,
) -> t.Tensor:
    """Scan chunks while carrying the final state into the next chunk."""
    raise NotImplementedError()


tests.test_chunked_affine_scan_state_carry(sequential_affine_scan, chunked_affine_scan)


### Exercise 5 - make the scan selective

Build exact event-conditioned coefficients. `WRITE` uses `(a, b) = (0, value)`, `hold` and `READ` use `(1, 0)`, and `ERASE` uses `(0, 0)`. Set `C_t=1` only at reads.

These 0/1 gates are the exact ground truth organism. A finite real Mamba parameterization approximates these limits with learned input-dependent values.

<details><summary>Expected output</summary>

```text
All tests in `test_selective_copy_ground_truth` passed!
```

The exact state trajectory is `[1, 1, 1, 1, 1, -0.7, -0.7, -0.7, -0.7, 0, 0, 0]`.
</details>

<details><summary>Help</summary>

Start with `a=1`, `b=0`, and `c=0`. Boolean masks for event IDs should modify only the corresponding scalar entries.
</details>

<details><summary>Solution</summary>

```python
write = event_ids.eq(EVENT_WRITE)
erase = event_ids.eq(EVENT_ERASE)
read = event_ids.eq(EVENT_READ)
a[..., 0][write | erase] = 0.0
b[..., 0][write] = values[write]
c[..., 0][read] = 1.0
return a, b, c
```
</details>


In [ ]:
def build_event_coefficients(
    event_ids: t.Tensor,
    values: t.Tensor,
) -> tuple[t.Tensor, t.Tensor, t.Tensor]:
    """Turn WRITE/HOLD/READ/ERASE events into exact a_t, b_t, and C_t gates."""
    raise NotImplementedError()


def selective_readout(states: t.Tensor, c: t.Tensor) -> t.Tensor:
    """Return y_t = sum_n C_t[n] h_t[n]."""
    raise NotImplementedError()


tests.test_selective_copy_ground_truth(
    build_event_coefficients,
    sequential_affine_scan,
    selective_readout,
)


### Exercise 6 - intervene on the recurrent state

Set the state to zero after position 6, then continue the original recurrence. This is a causal intervention: earlier reads must stay fixed, while the later read that depends on the ablated memory should change from `-0.7` to `0.0`.

<details><summary>Expected output</summary>

```text
All tests in `test_state_ablation_is_causal` passed!
```
</details>

<details><summary>Help</summary>

Scan through the intervention position, replace that post-update state, and use the replacement as the initial state for the untouched suffix recurrence.
</details>

<details><summary>Solution</summary>

```python
states = sequential_affine_scan(a[:, : position + 1], b[:, : position + 1], initial_state)
state = t.broadcast_to(t.as_tensor(replacement, dtype=b.dtype), states[:, -1].shape).clone()
states[:, -1] = state
for index in range(position + 1, a.shape[1]):
    state = a[:, index] * state + b[:, index]
    suffix.append(state)
```
</details>


In [ ]:
def intervene_on_state(
    a: t.Tensor,
    b: t.Tensor,
    position: int,
    replacement: t.Tensor | float = 0.0,
    initial_state: t.Tensor | None = None,
) -> t.Tensor:
    """Replace h_position, then continue the recurrence from that state."""
    raise NotImplementedError()


tests.test_state_ablation_is_causal(intervene_on_state)


## Signature Result: exact memory, failing controls, causal state

The figure is generated from the functions you implemented. Panel A makes every state transition inspectable. Panel B compares exact reads with two meaningful controls. Panel C tests whether the stored state causes the second answer.

The controls are:

- **fixed decay:** keep the same write impulses and read positions, but replace all input-conditioned transitions with `a_t=0.82`;
- **chunk reset:** use the correct selective updates but incorrectly reset state at each four-token boundary.

<details><summary>Expected output</summary>

```text
Sequential vs parallel max |diff|: 0.000e+00
Sequential vs chunked max |diff|:  0.000e+00
Selective read MAE:                0.000
Fixed-decay control MAE:           0.356
Chunk-reset control MAE:           0.233
Ablation effect at second read:    0.700
```
</details>

<details><summary>Interpreting the result</summary>

Exact sequential/parallel parity shows that the associative schedule implements the same recurrence. Zero selective read error shows that token-conditioned updates can isolate relevant values from distractors. The fixed-decay failure identifies selectivity as necessary for this exact task; the chunk-reset failure identifies state carry as necessary across boundaries. Finally, zeroing the state after the second write removes the second answer without changing the earlier read, locating the causal memory in `h_t`.
</details>


In [ ]:
def run_selective_copy_experiment(
    *,
    case: SelectiveCopyCase | None = None,
    fixed_decay: float = 0.82,
    chunk_size: int = 4,
    ablate_position: int = 6,
) -> SelectiveCopyResult:
    case = make_selective_copy_case() if case is None else case
    a, b, c = build_event_coefficients(case.event_ids, case.values)
    sequential_states = sequential_affine_scan(a, b)
    parallel_states = parallel_affine_scan(a, b)
    chunked_states = chunked_affine_scan(a, b, chunk_size)
    selective_reads = selective_readout(sequential_states, c)[0, case.read_positions]

    fixed_a = t.full_like(a, fixed_decay)
    fixed_states = sequential_affine_scan(fixed_a, b)
    fixed_decay_reads = selective_readout(fixed_states, c)[0, case.read_positions]

    reset_states = chunked_affine_scan(a, b, chunk_size, reset_each_chunk=True)
    reset_chunk_reads = selective_readout(reset_states, c)[0, case.read_positions]

    ablated_states = intervene_on_state(a, b, ablate_position, replacement=0.0)
    ablated_reads = selective_readout(ablated_states, c)[0, case.read_positions]

    return SelectiveCopyResult(
        case=case,
        a=a,
        b=b,
        c=c,
        sequential_states=sequential_states,
        parallel_states=parallel_states,
        chunked_states=chunked_states,
        selective_reads=selective_reads,
        fixed_decay_reads=fixed_decay_reads,
        reset_chunk_reads=reset_chunk_reads,
        ablated_reads=ablated_reads,
        parity_max_abs_diff=(sequential_states - parallel_states).abs().max().item(),
        chunked_max_abs_diff=(sequential_states - chunked_states).abs().max().item(),
        selective_read_mae=(selective_reads - case.read_targets).abs().mean().item(),
        fixed_decay_read_mae=(fixed_decay_reads - case.read_targets).abs().mean().item(),
        reset_chunk_read_mae=(reset_chunk_reads - case.read_targets).abs().mean().item(),
        ablation_effect=(selective_reads[1] - ablated_reads[1]).abs().item(),
    )


def render_signature_figure(
    result: SelectiveCopyResult,
    save_path: Path | None = None,
):
    positions = list(range(len(result.case.labels)))
    state = result.sequential_states[0, :, 0].detach().cpu()
    parallel = result.parallel_states[0, :, 0].detach().cpu()
    reads = result.case.read_positions.tolist()
    read_labels = [f"read @{position}" for position in reads]

    fig, axes = plt.subplots(3, 1, figsize=(12, 11), constrained_layout=True)
    event_colors = {EVENT_WRITE: "#e07a3f", EVENT_READ: "#2a9d6f", EVENT_ERASE: "#c44536"}
    for position, event in enumerate(result.case.event_ids[0].tolist()):
        if event in event_colors:
            axes[0].axvspan(position - 0.42, position + 0.42, color=event_colors[event], alpha=0.10)

    axes[0].step(positions, state, where="mid", color="#1f5a94", linewidth=2.5, label="sequential state")
    axes[0].scatter(positions, parallel, color="#111111", s=28, marker="x", label="parallel state")
    axes[0].axhline(0.0, color="#777777", linewidth=0.8)
    axes[0].set_ylabel("state h_t")
    axes[0].set_title("A. The scalar state writes, holds, overwrites, and erases")
    axes[0].set_xticks(positions, result.case.labels, rotation=40, ha="right")
    axes[0].legend(loc="upper right")

    width = 0.2
    x = t.arange(len(reads), dtype=t.float64)
    series = (
        (result.case.read_targets, "ground truth", "#222222"),
        (result.selective_reads, "selective", "#2a9d6f"),
        (result.fixed_decay_reads, "fixed decay", "#e07a3f"),
        (result.reset_chunk_reads, "reset each chunk", "#8f5aa8"),
    )
    for offset, (values, label, color) in enumerate(series):
        axes[1].bar(x + (offset - 1.5) * width, values.detach().cpu(), width, label=label, color=color)
    axes[1].axhline(0.0, color="#777777", linewidth=0.8)
    axes[1].set_xticks(x, read_labels)
    axes[1].set_ylabel("read output")
    axes[1].set_title("B. Selectivity matches exact reads; both controls fail")
    axes[1].legend(ncols=2)

    axes[2].bar(x - 0.18, result.selective_reads.detach().cpu(), 0.36, label="baseline", color="#1f5a94")
    axes[2].bar(x + 0.18, result.ablated_reads.detach().cpu(), 0.36, label="state set to zero after second write", color="#c44536")
    axes[2].axhline(0.0, color="#777777", linewidth=0.8)
    axes[2].set_xticks(x, read_labels)
    axes[2].set_ylabel("read output")
    axes[2].set_title("C. A state ablation removes only the downstream stored value")
    axes[2].legend()

    fig.suptitle(
        "Selective scan exact-memory organism\n"
        f"sequential/parallel max |diff| = {result.parity_max_abs_diff:.1e}; "
        f"selective read MAE = {result.selective_read_mae:.1e}",
        fontsize=14,
    )
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()
    return fig


result = run_selective_copy_experiment()
tests.test_signature_controls_are_informative(run_selective_copy_experiment)
print(f"Sequential vs parallel max |diff|: {result.parity_max_abs_diff:.3e}")
print(f"Sequential vs chunked max |diff|:  {result.chunked_max_abs_diff:.3e}")
print(f"Selective read MAE:                {result.selective_read_mae:.3f}")
print(f"Fixed-decay control MAE:           {result.fixed_decay_read_mae:.3f}")
print(f"Chunk-reset control MAE:           {result.reset_chunk_read_mae:.3f}")
print(f"Ablation effect at second read:    {result.ablation_effect:.3f}")

render_signature_figure(result)


## Try It Yourself

Change the two write values, any delay length, the fixed-decay strength, the chunk size, or the ablation position. Before running, predict which bars should move. A correct selective scan should keep zero read error for every delay; the fixed-decay control should worsen with longer delays; the reset control should fail only when a required memory crosses a boundary.


In [ ]:
# Change these values, delay lengths, control strength, chunk size, and intervention position.
play_case = make_selective_copy_case(
    first_value=1.6,
    second_value=-1.1,
    first_delay=4,
    second_delay=5,
    final_delay=2,
)
second_write = play_case.event_ids[0].eq(EVENT_WRITE).nonzero(as_tuple=False).flatten()[1].item()
play_result = run_selective_copy_experiment(
    case=play_case,
    fixed_decay=0.90,
    chunk_size=5,
    ablate_position=second_write + 2,
)
render_signature_figure(play_result)


## Bonus anomaly hunt: when does numerical parity drift?

Associativity is exact algebraically, but floating-point multiplication and addition are order-dependent. The cell below compares float32 and float64 over longer scans. Try moving `a` closer to `1`, increasing the scale of `b`, or using alternating signs. Find the smallest case where float32 exceeds `1e-5`, then inspect whether the error is concentrated after a large state excursion.

This is an anomaly hunt, not evidence that one execution path is semantically different: rerun in float64 and compare both against a higher-precision sequential reference before drawing that conclusion.


In [ ]:
def parity_error_at_length(length: int, dtype: t.dtype) -> float:
    generator = t.Generator().manual_seed(length)
    a = 0.90 + 0.099 * t.rand((1, length, 2), generator=generator, dtype=dtype)
    b = 0.05 * t.randn((1, length, 2), generator=generator, dtype=dtype)
    sequential = sequential_affine_scan(a, b)
    parallel = parallel_affine_scan(a, b)
    return (sequential - parallel).abs().max().item()


for dtype in (t.float32, t.float64):
    errors = {length: parity_error_at_length(length, dtype) for length in (16, 64, 256, 1024)}
    print(dtype, errors)


## Connection to Mamba

The [Mamba paper](https://arxiv.org/abs/2312.00752) makes SSM parameters input-dependent and uses a hardware-aware parallel scan during training. The [official implementation](https://github.com/state-spaces/mamba) contains optimized kernels for the same recurrence family. Our `WRITE`, `hold`, `READ`, and `ERASE` labels are semantic names for exact coefficient choices; a released language model receives no such labels and learns approximate coefficients from hidden representations.

The important transfer is the contract you tested: discretization produces affine updates, associative composition preserves their order, recurrent inference carries state, and intervention on that state can change later outputs.

## Limitations

- The organism has one scalar state and hand-specified exact gates; it is not a trained language model.
- Finite Mamba parameters cannot produce exact `a=0` or `a=1`; these are clean limiting cases used to expose the recurrence.
- The Python Hillis-Steele scan demonstrates semantics, not kernel speed or memory efficiency.
- The intervention establishes causality for this known state variable, not a general method for interpreting distributed states in a large Mamba model.
- CPU parity at these lengths does not validate CUDA fused kernels or released-checkpoint weight mapping.

## Reading links

- [Mamba: Linear-Time Sequence Modeling with Selective State Spaces](https://arxiv.org/abs/2312.00752)
- [The Annotated S4](https://srush.github.io/annotated-s4/), for the continuous-to-discrete SSM background
- [state-spaces/mamba](https://github.com/state-spaces/mamba), for the optimized reference implementation
